In [6]:
import kagglehub
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Download dataset
path = kagglehub.dataset_download("shrutimechlearn/churn-modelling")
print("Dataset downloaded to:", path)

# Find CSV file
csv_file = None
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_file = os.path.join(root, file)
            break

print("CSV File:", csv_file)

# Load dataset
df = pd.read_csv(csv_file)

# Drop unnecessary columns
df = df.drop(
    columns=[
        "RowNumber",
        "CustomerId",
        "Surname"
    ]
)

# Features and target
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Encode categorical features
gender_encoder = LabelEncoder()
geo_encoder = LabelEncoder()

X["Gender"] = gender_encoder.fit_transform(X["Gender"])
X["Geography"] = geo_encoder.fit_transform(X["Geography"])

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ANN Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    
    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),
    
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        32,
        activation="relu"
    ),

    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(
        16,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        1,
        activation="sigmoid"
    )
])

# Compile
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"\nTest Accuracy: {accuracy:.4f}")

# Save artifacts
os.makedirs("../models", exist_ok=True)

model.save("../models/ann_churn.keras")

with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("../models/geography_encoder.pkl", "wb") as f:
    pickle.dump(geo_encoder, f)

with open("../models/gender_encoder.pkl", "wb") as f:
    pickle.dump(gender_encoder, f)

print("\nSaved:")
print("models/ann_churn.keras")
print("models/scaler.pkl")
print("models/geography_encoder.pkl")
print("models/gender_encoder.pkl")

100%|██████████| 262k/262k [00:01<00:00, 252kB/s]

Extracting files...
Dataset downloaded to: C:\Users\vamsh\.cache\kagglehub\datasets\shrutimechlearn\churn-modelling\versions\1
CSV File: C:\Users\vamsh\.cache\kagglehub\datasets\shrutimechlearn\churn-modelling\versions\1\Churn_Modelling.csv
Epoch 1/100


200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7812 - loss: 0.5018 - val_accuracy: 0.8012 - val_loss: 0.4424
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8030 - loss: 0.4445 - val_accuracy: 0.8219 - val_loss: 0.4258
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8091 - loss: 0.4290 - val_accuracy: 0.8469 - val_loss: 0.4049
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8198 - loss: 0.4162 - val_accuracy: 0.8537 - val_loss: 0.3831
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8280 - loss: 0.4013 - val_accuracy: 0.8631 - val_loss: 0.3646
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8363 - loss: 0.3932 - val_accuracy: 0.8637 - val_loss: 0.3553
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8380 - loss: 0.3857 - val_accuracy: 0.8669 - val_loss: 0.3484
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8444 - loss: 0.3765 - val_accuracy: 0.8669